# Reverse Model: Fit Physics Parameters from (RPM, Distance) Data

This notebook takes measured **(RPM, distance)** pairs from a topspin flywheel shooter,
fits the physics model parameters to best match the data, and then uses the fitted model
to predict **how long it takes for the ball to reach a given target height**.

The distance is the horizontal distance from the shooter to where the ball lands
at the target height (on descent).

### Parameters
- **Fitted**: `N` (efficiency) and `Cd` (drag coefficient, constrained near 0.47)
- **Fixed**: `Cl` is always computed as `0.2 * R_FUEL * W0 / V0` (standard topspin formula)

## How to use
1. Enter your measured `(RPM, distance)` pairs in the **Data Input** cell below.
2. Adjust `TARGET_HEIGHT` to the height you care about (in meters).
3. Run all cells.

In [1]:
import math
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

ModuleNotFoundError: No module named 'numpy'

---
## ✏️ Data Input & Configuration
Edit the cells below to match your setup.

In [ ]:
# ──────────────────────────────────────────────
# Measured (RPM, distance_in_meters) pairs
# distance = horizontal distance from the
#            shooter to where the ball lands.
# Replace / extend with your own measurements.
# ──────────────────────────────────────────────
measured_data = [
    (2600, 1.106),
    (2800, 1.264),
    (3000, 1.430),
    (3200, 1.601),
    (3400, 1.778),
    (3600, 1.958),
    (3800, 2.142),
    (4000, 2.329),
    (4200, 2.518),
    (4400, 2.707),
    (4600, 2.897),
    (4800, 3.087),
    (5000, 3.276),
    (5200, 3.464),
    (5400, 3.650),
    (5600, 3.833),
]

rpm_measured  = np.array([d[0] for d in measured_data], dtype=float)
dist_measured = np.array([d[1] for d in measured_data], dtype=float)

print(f"Loaded {len(measured_data)} data points.")
print(f"RPM range : {rpm_measured.min():.0f} – {rpm_measured.max():.0f}")
print(f"Dist range: {dist_measured.min():.3f} – {dist_measured.max():.3f} m")

In [ ]:
# ──────────────────────────────────────────────
# Target height (meters)  ← CHANGE THIS EASILY
# This is the height at which you want to know
# how long the ball takes to arrive.
# ──────────────────────────────────────────────
TARGET_HEIGHT = 1.872  # meters (e.g. top of the goal)

print(f"Target height: {TARGET_HEIGHT} m")

In [ ]:
# ──────────────────────────────────────────────
# Fixed / known physical constants
# Adjust if your hardware differs.
# ──────────────────────────────────────────────
R_FUEL  = 0.15 / 2    # ball radius (m)
D_WHEEL = 0.1016      # flywheel diameter (m)
MASS    = 0.20366297   # ball mass (kg)
G       = 9.81         # gravity (m/s²)
RHO     = 1.225        # air density (kg/m³)
A_CROSS = math.pi * R_FUEL**2  # cross-sectional area

# Base lift coefficient (used in Cl = CL_BASE * R_FUEL * W0 / V0)
CL_BASE = 0.2

# Launch angle (degrees) – change if your shooter has a different angle
LAUNCH_ANGLE_DEG = 72
THETA = math.radians(LAUNCH_ANGLE_DEG)

# Initial height of the launcher exit (m)
Y0 = 0.2032

# Simulation settings
DELTA_T  = 0.00001   # time step (s)
MAX_TIME = 3.0       # max simulation duration (s)

print("Physical constants loaded.")

---
## Forward Simulation Engine

A single-RPM trajectory simulator that returns the **distance** where the ball
crosses `target_height` on descent, and the **time** at which that happens.

`Cl` is always computed from the standard topspin formula: `CL_BASE * R_FUEL * W0 / V0`.

In [ ]:
def simulate_single(
    rpm: float,
    N: float,
    Cd: float,
    target_height: float,
):
    """
    Simulate a single trajectory for the given RPM and model parameters.

    Cl is computed internally as CL_BASE * R_FUEL * W0 / V0.

    Returns
    -------
    distance : float or None
        Horizontal distance where ball descends through target_height.
    time_s : float or None
        Time (seconds) at which that crossing happens.
    """
    # Exit velocity
    V0 = ((rpm * 2 * math.pi) / 60 * D_WHEEL / 2) / 2 * N
    if V0 <= 0:
        return None, None

    # Spin-adjusted lift coefficient (standard topspin formula)
    W0 = V0 / R_FUEL
    Cl = CL_BASE * R_FUEL * W0 / V0  # = CL_BASE for this geometry

    # Factor that appears repeatedly
    k = -0.5 * RHO * A_CROSS / MASS

    # Initial velocities
    vx = V0 * math.cos(THETA)
    vy = V0 * math.sin(THETA)

    # Initial position
    cx, cy = 0.0, Y0

    target_with_radius = target_height + R_FUEL

    t = 0.0
    while t < MAX_TIME:
        speed = math.sqrt(vx**2 + vy**2)

        # Accelerations (topspin: spin = +1)
        ax = k * speed * (Cd * vx + Cl * vy)
        ay = -G + k * speed * (Cd * vy - Cl * vx)

        # Update velocity
        vx += ax * DELTA_T
        vy += ay * DELTA_T

        # Update position
        cx += vx * DELTA_T
        cy += vy * DELTA_T

        t += DELTA_T

        # Check if ball is descending through target height
        if cy > target_with_radius and vy < 0:
            return cx, t

    return None, None  # didn't reach target

---
## Parameter Fitting

We fit **N** (efficiency) and **Cd** (drag coefficient, constrained near 0.47)
so that the simulated distances best match the measured data (least-squares).

`Cl` is **not** a free parameter — it's always `0.2 * R_FUEL * W0 / V0`.

In [ ]:
def objective(params):
    """
    Sum of squared errors between simulated and measured distances.
    """
    N, Cd = params
    total_err = 0.0
    for rpm, d_meas in zip(rpm_measured, dist_measured):
        d_sim, _ = simulate_single(rpm, N, Cd, TARGET_HEIGHT)
        if d_sim is None:
            total_err += 1e6   # heavy penalty if the ball never reaches target
        else:
            total_err += (d_sim - d_meas) ** 2
    return total_err


# Initial guess
x0 = [0.92, 0.47]

# Parameter bounds
# Cd is tightly constrained around 0.47
bounds = [
    (0.5, 1.5),     # N  – efficiency
    (0.35, 0.60),   # Cd – drag coefficient (near 0.47)
]

print("Starting parameter optimisation (this may take a minute) …")
result = minimize(objective, x0, method='Nelder-Mead', bounds=bounds,
                  options={'maxiter': 2000, 'xatol': 1e-6, 'fatol': 1e-8})

N_fit, Cd_fit = result.x

print(f"\n✅ Optimisation finished  (success={result.success})")
print(f"   N     = {N_fit:.6f}")
print(f"   Cd    = {Cd_fit:.6f}")
print(f"   Cl    = {CL_BASE} * R_FUEL * W0 / V0  (fixed formula)")
print(f"   Residual SSE = {result.fun:.8f}")

---
## Model vs. Data Comparison

In [ ]:
# Simulate across a range of RPMs with fitted parameters
rpm_range = np.arange(rpm_measured.min(), rpm_measured.max() + 1, 100)

dist_sim = []
for rpm in rpm_range:
    d, _ = simulate_single(rpm, N_fit, Cd_fit, TARGET_HEIGHT)
    dist_sim.append(d)

# Also get per-measurement-point predictions
dist_pred = []
for rpm in rpm_measured:
    d, _ = simulate_single(rpm, N_fit, Cd_fit, TARGET_HEIGHT)
    dist_pred.append(d)

plt.figure(figsize=(10, 6))
plt.scatter(rpm_measured, dist_measured, color='black', s=60, zorder=5, label='Measured data')
plt.plot(rpm_range, dist_sim, 'r-', linewidth=2, label='Fitted model')
plt.xlabel('RPM', fontsize=13)
plt.ylabel('Distance (m)', fontsize=13)
plt.title(f'Model Fit  –  N={N_fit:.4f}, Cd={Cd_fit:.4f}, Cl=0.2·R·ω/v', fontsize=14)
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Print per-point residuals
print(f"\n{'RPM':>6}  {'Measured':>10}  {'Predicted':>10}  {'Error':>10}")
print('-' * 42)
for rpm, dm, dp in zip(rpm_measured, dist_measured, dist_pred):
    err = (dp - dm) if dp is not None else float('nan')
    print(f"{rpm:6.0f}  {dm:10.4f}  {dp if dp is not None else 'N/A':>10}  {err:10.4f}")

---
## ⏱ Time-to-Target-Height Lookup

Using the fitted parameters, compute how long (in seconds) the ball takes
to reach `TARGET_HEIGHT` for every RPM in the data set and beyond.

In [ ]:
# ──────────────────────────────────────────────
# RPM range for the lookup table
# Change these to whatever range you need.
# ──────────────────────────────────────────────
LOOKUP_RPM_START = int(rpm_measured.min())
LOOKUP_RPM_END   = int(rpm_measured.max())
LOOKUP_RPM_STEP  = 100

print(f"Target height: {TARGET_HEIGHT} m")
print(f"{'RPM':>6}  {'Distance (m)':>13}  {'Time (s)':>10}")
print('-' * 34)

lookup_rpms  = []
lookup_times = []
lookup_dists = []

for rpm in range(LOOKUP_RPM_START, LOOKUP_RPM_END + 1, LOOKUP_RPM_STEP):
    d, t = simulate_single(rpm, N_fit, Cd_fit, TARGET_HEIGHT)
    if d is not None:
        print(f"{rpm:6d}  {d:13.4f}  {t:10.6f}")
        lookup_rpms.append(rpm)
        lookup_times.append(t)
        lookup_dists.append(d)
    else:
        print(f"{rpm:6d}  {'N/A':>13}  {'N/A':>10}")

In [ ]:
# Plot RPM vs Time-to-target
plt.figure(figsize=(10, 6))
plt.plot(lookup_rpms, lookup_times, 'bo-', linewidth=2, markersize=5)
plt.xlabel('RPM', fontsize=13)
plt.ylabel('Time to target (s)', fontsize=13)
plt.title(f'Time to reach {TARGET_HEIGHT} m  (fitted model)', fontsize=14)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 🔍 Quick Single-RPM Query

Change `QUERY_RPM` below to check a specific RPM.

In [ ]:
QUERY_RPM = 4000  # ← change me

d, t = simulate_single(QUERY_RPM, N_fit, Cd_fit, TARGET_HEIGHT)
if d is not None:
    print(f"At {QUERY_RPM} RPM the ball reaches {TARGET_HEIGHT} m height")
    print(f"  → Distance : {d:.4f} m")
    print(f"  → Time     : {t:.6f} s  ({t*1000:.2f} ms)")
else:
    print(f"At {QUERY_RPM} RPM the ball never reaches {TARGET_HEIGHT} m.")